# Hunyuan3D-2.1 — 3D Model Generation
Generate 3D models from images using Tencent's Hunyuan3D-2.1.

**Setup**: Runtime > Change runtime type > **T4 GPU** (free) or **A100** (Pro)

In [1]:
# Cell 1: Check GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

Tesla T4, 15360 MiB


In [2]:
# Cell 2: Clone and install Hunyuan3D-2.1
!rm -rf /content/Hunyuan3D-2.1
!git clone https://github.com/Tencent-Hunyuan/Hunyuan3D-2.1.git /content/Hunyuan3D-2.1
%cd /content/Hunyuan3D-2.1

# Install curated deps (skip requirements.txt — has Chinese mirrors + broken builds)
!pip install -q \
  ninja pybind11 \
  transformers==4.46.0 diffusers==0.30.0 accelerate huggingface-hub safetensors \
  numpy scipy einops \
  opencv-python imageio scikit-image \
  rembg trimesh pygltflib xatlas \
  omegaconf pyyaml tqdm \
  pytorch-lightning torchmetrics timm \
  onnxruntime torchdiffeq pydantic \
  gradio

# Build custom_rasterizer (CUDA extension)
!cd hy3dpaint/custom_rasterizer && pip install -e . --no-build-isolation -q

# JIT compile mesh_inpaint_processor (C++ only, no setup.py)
!cd /content/Hunyuan3D-2.1/hy3dpaint/DifferentiableRenderer && python -c "import torch; from torch.utils.cpp_extension import load; load(name='mesh_inpaint_processor', sources=['mesh_inpaint_processor.cpp'], verbose=True)"

# Patch mesh_utils.py — make bpy import optional (trimesh fallback)
import pathlib
mu = pathlib.Path('hy3dpaint/DifferentiableRenderer/mesh_utils.py')
src = mu.read_text()
src = src.replace(
    'import bpy',
    'try:\n    import bpy\n    HAS_BPY = True\nexcept ImportError:\n    HAS_BPY = False'
)
mu.write_text(src)

print('Installation complete!')

Cloning into '/content/Hunyuan3D-2.1'...
remote: Enumerating objects: 974, done.
remote: Counting objects: 100% (292/292), done.
remote: Compressing objects: 100% (128/128), done.
remote: Total 974 (delta 189), reused 164 (delta 164), pack-reused 682 (from 1)
Receiving objects: 100% (974/974), 148.92 MiB | 36.73 MiB/s, done.
Resolving deltas: 100% (263/263), done.
Updating files: 100% (598/598), done.
/content/Hunyuan3D-2.1
Reason for being yanked: This version unfortunately does not work with 3.8 but we did not drop the support yet
  Checking if build backend supports build_editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for custom_rasterizer (pyproject.toml) ... done
[1/2] c++ -MMD -MF mesh_inpaint_processor.o.d -DTORCH_EXTENSION_NAME=mesh_inpaint_processor -DTORCH_API_INCLUDE_EXTENSION_H -isystem /usr/local/lib/python3.13/dist-packages/torch/include -isystem /usr/local/lib/python3.13/dist-packages/torch/include/torch/csrc/api/include -is

In [2]:
# Cell 3: Upload your input image
from google.colab import files
uploaded = files.upload()
input_image = list(uploaded.keys())[0]
print(f'Uploaded: {input_image}')

Saving IMG_0972.JPG to IMG_0972 (1).JPG
Uploaded: IMG_0972 (1).JPG


In [4]:
import sys
sys.path.insert(0, '/content/Hunyuan3D-2.1/hy3dshape')

In [ ]:
# Cell 4: Stage 1 — Shape Generation
import sys, os
sys.path.insert(0, 'hy3dshape')
sys.path.insert(0, 'hy3dpaint')

from PIL import Image
from hy3dshape import Hunyuan3DDiTFlowMatchingPipeline
from hy3dshape.rembg import BackgroundRemover

print('Loading shape pipeline...')
pipeline = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained(
    'tencent/Hunyuan3D-2.1', subfolder='hunyuan3d-dit-v2-1')

image = Image.open(input_image)
image = BackgroundRemover()(image)

print('Generating shape...')
mesh = pipeline(
    image=image,
    num_inference_steps=30,
    guidance_scale=7.5,
    octree_resolution=256,
)[0]

os.makedirs('output', exist_ok=True)
mesh.export('output/shape.glb')
print('Shape saved: output/shape.glb')

/usr/local/lib/python3.13/dist-packages/numba/np/ufunc/parallel.py:373: NumbaWarning: The TBB threading layer requires TBB version 2021 update 6 or later i.e., TBB_INTERFACE_VERSION >= 12060. Found TBB_INTERFACE_VERSION = 12050. The TBB threading layer is disabled.
  warnings.warn(problem)
2026-11-12 15:10:32,966 - hy3dgen.shapgen - INFO - Try to load model from local path: /root/.cache/hy3dgen/tencent/Hunyuan3D-2.1/hunyuan3d-dit-v2-1
INFO:hy3dgen.shapgen:Try to load model from local path: /root/.cache/hy3dgen/tencent/Hunyuan3D-2.1/hunyuan3d-dit-v2-1
2026-11-12 15:10:32,975 - hy3dgen.shapgen - INFO - Loading model from /root/.cache/hy3dgen/tencent/Hunyuan3D-2.1/hunyuan3d-dit-v2-1/model.fp16.ckpt
INFO:hy3dgen.shapgen:Loading model from /root/.cache/hy3dgen/tencent/Hunyuan3D-2.1/hunyuan3d-dit-v2-1/model.fp16.ckpt


Loading shape pipeline...


In [1]:
%cd /content/Hunyuan3D-2.1
!pip install -e hy3dshape

/content/Hunyuan3D-2.1
Obtaining file:///content/Hunyuan3D-2.1/hy3dshape
ERROR: file:///content/Hunyuan3D-2.1/hy3dshape does not appear to be a Python project: neither 'setup.py' nor 'pyproject.toml' found.


In [1]:
!pip install -U pillow pymeshlab

In [2]:
%cd /content/Hunyuan3D-2.1/hy3dshape


/content/Hunyuan3D-2.1/hy3dshape


In [ ]:
# Cell 5: Stage 2 — Texture Generation
from textureGenPipeline import Hunyuan3DPaintPipeline, Hunyuan3DPaintConfig

print('Loading texture pipeline...')
conf = Hunyuan3DPaintConfig(6, 512)
conf.realesrgan_ckpt_path = 'hy3dpaint/ckpt/RealESRGAN_x4plus.pth'
conf.multiview_cfg_path = 'hy3dpaint/cfgs/hunyuan-paint-pbr.yaml'
conf.custom_pipeline = 'hy3dpaint/hunyuanpaintpbr'
paint = Hunyuan3DPaintPipeline(conf)

print('Generating texture...')
paint(
    mesh_path='output/shape.glb',
    image_path=image,
    output_mesh_path='output/textured.obj',
    save_glb=False,
)
print('Textured model saved: output/textured.obj')

In [1]:
# Cell 6: Convert to GLB and visualize
import trimesh
mesh = trimesh.load('output/textured.obj', process=False)
mesh.export('output/textured.glb', file_type='glb')
print('GLB saved: output/textured.glb')

# Show in notebook
mesh.show()

ValueError: string is not a file: `output/textured.obj`

In [ ]:
# Cell 7: Download results
from google.colab import files
files.download('output/shape.glb')
files.download('output/textured.glb')
files.download('output/textured.obj')
files.download('output/textured.jpg')
print('Downloads started!')